# Módulo 1 · Clase 1 (Teoría) — Del perceptrón a los Transformers
### Deep Learning · Apunte de cátedra

> **Cómo usar este notebook (profesor):** úsalo como apunte vivo. Las secciones en *markdown* son para explicar; las celdas de código son **demos cortas** para ejecutar en vivo. Las imágenes son tus slides, embebidas por URL desde el repo.

**Objetivos de la clase.** Al terminar, los estudiantes deberían poder:
1. Explicar qué es Deep Learning y sus cuatro ingredientes (datos, modelo, pérdida, optimización).
2. Entender por qué un perceptrón no basta y cómo un MLP lo resuelve.
3. Describir cómo aprende una red: pérdida → gradiente descendente → backpropagation.
4. Reconocer las decisiones de optimización y regularización que hacen que el entrenamiento funcione.
5. Tener una primera intuición de la **atención** y los Transformers.

**Agenda (≈ 3 horas, con un descanso a la mitad):**
| Bloque | Tema | ~min |
|---|---|---|
| 0 | ¿Qué es Deep Learning? | 20 |
| 1 | Perceptrón y su límite | 25 |
| 2 | Perceptrón multicapa (MLP) | 30 |
| 3 | Cómo aprende: funciones de pérdida | 20 |
| — | *Descanso* | 10 |
| 4 | Gradiente descendente | 25 |
| 5 | Backpropagation y autograd | 30 |
| 6 | Optimización y regularización | 25 |
| 7 | Panorama de arquitecturas + intro a atención | 25 |


In [ ]:
# Setup (ejecutar al inicio de la clase)
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(0); np.random.seed(0)
print("PyTorch:", torch.__version__, "| GPU disponible:", torch.cuda.is_available())

## Bloque 0 · ¿Qué es Deep Learning?

**Machine Learning clásico:** un humano diseña *features* y el modelo aprende a combinarlas.
**Deep Learning:** la red aprende **representaciones jerárquicas** directamente desde los datos crudos. Cada capa transforma la entrada en una representación más útil para la tarea.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/00_que_es_dl.png" width="760">

<sub>Deep Learning es un subconjunto del Machine Learning, que a su vez es un subconjunto de la IA.</sub>

### Los cuatro ingredientes
Toda red profunda se entrena con la misma receta:

1. **Datos** — pares $(x, y)$ (supervisado) o solo $x$ (no supervisado).
2. **Modelo** — una función $f_\theta(x)$ con parámetros $\theta$ (los pesos).
3. **Función de pérdida** $\mathcal{L}(f_\theta(x), y)$ — mide *qué tan mal* lo hace el modelo.
4. **Algoritmo de optimización** — ajusta $\theta$ para minimizar la pérdida (gradiente descendente).

> La idea central de todo el curso: **aprender = minimizar una pérdida moviendo los parámetros en la dirección que la baja**.

El resto del curso son *variaciones del modelo* (MLP, CNN, RNN, Transformer) sobre esta misma receta.

## Bloque 1 · El perceptrón

El perceptrón es la unidad básica: una **combinación lineal** de las entradas seguida de una función de activación.

$$ z = \mathbf{w}^\top \mathbf{x} + b, \qquad \hat{y} = \phi(z) $$

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/01_perceptron.png" width="760">

<sub>Una neurona: entradas ponderadas por pesos, sumadas con un bias y pasadas por una función de activación.</sub>

Con $\phi$ = función escalón, separa el espacio con un **hiperplano**. Sirve para problemas *linealmente separables*.

**Demo:** una neurona aprende la frontera de un problema separable (AND).

In [ ]:
# Demo: perceptrón resolviendo AND (linealmente separable)
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([0,0,0,1], dtype=float)  # AND

w = np.zeros(2); b = 0.0; lr = 0.5
for epoch in range(20):
    for xi, yi in zip(X, y):
        pred = 1.0 if (w @ xi + b) > 0 else 0.0
        err = yi - pred
        w += lr * err * xi
        b += lr * err
print("Pesos aprendidos:", w, "bias:", b)
print("Predicciones:", [1 if (w@xi+b)>0 else 0 for xi in X], "esperado:", list(y.astype(int)))

### El límite del perceptrón: XOR
No todos los problemas son linealmente separables. El caso clásico es **XOR**: ninguna línea recta separa las clases. Un solo perceptrón **no puede** resolverlo.

In [ ]:
# Visualización: XOR no es separable por una recta
Xx = np.array([[0,0],[0,1],[1,0],[1,1]]); yx = np.array([0,1,1,0])  # XOR
plt.figure(figsize=(4,4))
for cls, marker in [(0,'o'),(1,'s')]:
    pts = Xx[yx==cls]
    plt.scatter(pts[:,0], pts[:,1], marker=marker, s=200, label=f"clase {cls}")
plt.title("XOR: ninguna recta separa los cuadrados de los círculos")
plt.xlabel("x1"); plt.ylabel("x2"); plt.legend(); plt.grid(True); plt.show()

## Bloque 2 · Perceptrón multicapa (MLP)

La solución: **apilar** neuronas en capas e intercalar **no-linealidades**. Una red de 2 capas ya resuelve XOR.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/02_mlp.png" width="760">

<sub>Arquitectura de un MLP: capa de entrada, una o más capas ocultas y una capa de salida.</sub>

$$ \mathbf{h} = \phi(W_1 \mathbf{x} + \mathbf{b}_1), \qquad \hat{y} = \phi(W_2 \mathbf{h} + \mathbf{b}_2) $$

La **no-linealidad** $\phi$ es esencial: sin ella, componer capas lineales da otra capa lineal (no ganamos nada).

### Funciones de activación
- **Sigmoid** $\;\sigma(z)=\frac{1}{1+e^{-z}}$ — satura, gradientes pequeños en los extremos.
- **Tanh** — centrada en 0, también satura.
- **ReLU** $\;\max(0,z)$ — barata, no satura para $z>0$, es la opción por defecto hoy.

In [ ]:
# Demo: funciones de activación
z = np.linspace(-6, 6, 200)
sigmoid = 1/(1+np.exp(-z)); tanh = np.tanh(z); relu = np.maximum(0, z)
fig, ax = plt.subplots(1,3, figsize=(12,3))
for a,(name,vals) in zip(ax, [("sigmoid",sigmoid),("tanh",tanh),("ReLU",relu)]):
    a.plot(z, vals); a.set_title(name); a.grid(True); a.axhline(0,color='k',lw=.5); a.axvline(0,color='k',lw=.5)
plt.tight_layout(); plt.show()

### Teorema de aproximación universal (intuición)
Un MLP con **una capa oculta suficientemente ancha** puede aproximar *cualquier* función continua en un dominio acotado. En la práctica, redes **más profundas** logran lo mismo con muchos menos parámetros: por eso *deep* learning.

**Demo:** un MLP de 1 capa oculta sí resuelve XOR.

In [ ]:
# Demo: MLP minimalista resuelve XOR (PyTorch)
Xt = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
yt = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)  # XOR

model = nn.Sequential(nn.Linear(2,8), nn.ReLU(), nn.Linear(8,1))
opt = torch.optim.Adam(model.parameters(), lr=0.1)
loss_fn = nn.BCEWithLogitsLoss()
for step in range(500):
    opt.zero_grad()
    loss = loss_fn(model(Xt), yt)
    loss.backward(); opt.step()
pred = (torch.sigmoid(model(Xt)) > 0.5).int().flatten().tolist()
print("Predicción XOR:", pred, "| esperado: [0, 1, 1, 0] | loss final:", round(loss.item(),4))

## Bloque 3 · Cómo aprende: funciones de pérdida

La pérdida convierte "qué tan mal lo hace el modelo" en un número que podemos minimizar.

**Regresión (salida continua): error cuadrático medio (MSE)**
$$ \mathcal{L}_{\text{MSE}} = \frac{1}{N}\sum_i (\hat{y}_i - y_i)^2 $$

**Clasificación multiclase: softmax + cross-entropy.** La red produce *logits* $z$; softmax los convierte en una distribución de probabilidad.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/03_softmax.png" width="760">

<sub>Softmax transforma logits en una distribución de probabilidad: exponencia y normaliza para que sumen 1.</sub>

$$ p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}, \qquad \mathcal{L}_{\text{CE}} = -\sum_k y_k \log p_k $$
Con etiqueta verdadera $c$, la pérdida es simplemente $-\log p_c$: penaliza fuerte cuando el modelo está *confiadamente equivocado*.

In [ ]:
# Demo: softmax y cross-entropy
logits = np.array([2.0, 0.5, -1.0])  # salida cruda para 3 clases
probs = np.exp(logits) / np.exp(logits).sum()
print("logits:", logits)
print("probabilidades (softmax):", np.round(probs, 3), "| suman:", round(probs.sum(),3))
clase_correcta = 0
print(f"cross-entropy si la clase correcta es {clase_correcta}: {-np.log(probs[clase_correcta]):.3f}")
print(f"cross-entropy si la clase correcta fuera 2 (mal predicha): {-np.log(probs[2]):.3f}")

## Bloque 4 · Gradiente descendente

Queremos los $\theta$ que minimizan $\mathcal{L}(\theta)$. El **gradiente** $\nabla_\theta \mathcal{L}$ apunta hacia donde la pérdida *sube* más rápido; nos movemos en sentido contrario:

$$ \theta \leftarrow \theta - \eta \, \nabla_\theta \mathcal{L} $$

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/04_gradiente_descendente.png" width="760">

<sub>Aprender = encontrar el mínimo de la función de costo. Con millones de parámetros, lo hacemos por gradiente descendente, no analíticamente.</sub>

$\eta$ es el **learning rate**: muy chico → lento; muy grande → diverge.

**Demo:** descenso sobre una función 1D, viendo el efecto del learning rate.

In [ ]:
# Demo: gradiente descendente sobre f(x) = x^2 con distintos learning rates
def f(x): return x**2
def grad(x): return 2*x

fig, axes = plt.subplots(1,3, figsize=(12,3.2))
for ax, lr in zip(axes, [0.1, 0.5, 1.01]):
    xs = np.linspace(-3,3,100); ax.plot(xs, f(xs), 'lightgray')
    x = 2.5; traj=[x]
    for _ in range(12):
        x = x - lr*grad(x); traj.append(x)
    traj=np.array(traj); ax.plot(traj, f(traj), 'o-', ms=4)
    ax.set_title(f"lr={lr}"); ax.grid(True)
axes[0].set_ylabel("f(x)=x²")
plt.suptitle("lr chico: converge lento · lr ok: converge · lr grande: diverge")
plt.tight_layout(); plt.show()

**Variantes en la práctica:**
- **Batch GD:** usa todo el dataset por paso (caro).
- **SGD:** usa un ejemplo por paso (ruidoso pero rápido).
- **Mini-batch SGD:** lo estándar — un lote de 32–256 ejemplos por paso. Equilibra ruido y eficiencia, y aprovecha la GPU.

## Bloque 5 · Backpropagation

Para usar gradiente descendente necesitamos $\nabla_\theta \mathcal{L}$ de una red con millones de parámetros. **Backpropagation** = aplicar la **regla de la cadena** de forma eficiente sobre el **grafo computacional**: calculamos la salida hacia adelante (*forward*) y luego propagamos las derivadas hacia atrás (*backward*).

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/05_backprop.png" width="760">

<sub>El error se propaga hacia atrás capa por capa: el gradiente de una capa depende del de la capa siguiente.</sub>

Ejemplo de cadena: si $\mathcal{L} = (\hat y - y)^2$ y $\hat y = w x + b$, entonces
$$ \frac{\partial \mathcal{L}}{\partial w} = \underbrace{2(\hat y - y)}_{\partial \mathcal{L}/\partial \hat y}\cdot \underbrace{x}_{\partial \hat y/\partial w} $$

En la práctica **no derivamos a mano**: los frameworks tienen *autograd* (diferenciación automática).

In [ ]:
# Demo: autograd de PyTorch hace backprop por nosotros
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x, y = 2.0, 10.0

y_hat = w*x + b           # forward
loss = (y_hat - y)**2     # pérdida
loss.backward()           # backward: calcula gradientes

print(f"y_hat = {y_hat.item()}, loss = {loss.item()}")
print(f"dL/dw (autograd) = {w.grad.item()}  |  a mano: 2*(y_hat-y)*x = {2*(y_hat.item()-y)*x}")
print(f"dL/db (autograd) = {b.grad.item()}  |  a mano: 2*(y_hat-y)   = {2*(y_hat.item()-y)}")

## Bloque 6 · Optimización y regularización en la práctica

Entrenar redes profundas requiere más que SGD básico.

**Optimizadores:**
- **Momentum:** acumula una "velocidad" → atraviesa zonas planas y amortigua oscilaciones.
- **Adam:** adapta el learning rate por parámetro usando estimaciones del primer y segundo momento del gradiente. Es el *default* hoy.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/06_optimizadores.png" width="760">

<sub>Distintos optimizadores recorren la superficie de pérdida de forma diferente: momentum y Adam suelen converger más rápido y estable que SGD.</sub>

**Inicialización:** pesos mal inicializados → gradientes que explotan o desaparecen. **Xavier/He** escalan la varianza según el tamaño de la capa.
**Normalización (BatchNorm):** normaliza las activaciones por mini-batch → entrenamiento más estable y rápido.

**Regularización (combatir el overfitting):** **weight decay (L2)** penaliza pesos grandes; **dropout** apaga neuronas al azar en entrenamiento; **early stopping** detiene cuando la validación deja de mejorar.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/07_dropout.png" width="620">

<sub>Dropout: en cada paso se apagan neuronas al azar, forzando a la red a no depender de neuronas individuales.</sub>

> **Overfitting** = el modelo memoriza el train pero no generaliza. Se ve cuando la curva de *train* sigue bajando pero la de *validación* sube.

In [ ]:
# Demo: anatomía de una curva con overfitting (datos ilustrativos)
epochs = np.arange(1, 41)
train = 1.2*np.exp(-epochs/8) + 0.03
val   = 1.2*np.exp(-epochs/8) + 0.03 + 0.0009*(epochs-12)**2*(epochs>12)
plt.figure(figsize=(6,3.5))
plt.plot(epochs, train, label="train loss")
plt.plot(epochs, val, label="validation loss")
plt.axvline(15, ls='--', color='gray'); plt.text(15.5, 0.6, "early stopping", color='gray')
plt.xlabel("época"); plt.ylabel("loss"); plt.legend(); plt.title("Overfitting: val sube mientras train baja"); plt.grid(True); plt.show()

## Bloque 7 · Panorama de arquitecturas y primer contacto con la atención

El MLP trata cada entrada como un vector plano. Para datos con **estructura** usamos arquitecturas especializadas (los módulos que siguen):

- **CNN** (Módulo 2): explotan la estructura espacial de las imágenes con convoluciones (localidad + pesos compartidos).
- **RNN** (Módulo 3): procesan secuencias paso a paso, manteniendo un estado.
- **Transformers** (Módulos 3–4): reemplazan la recurrencia por **atención**, procesando toda la secuencia en paralelo. Son la base de los LLMs y de la IA moderna.

### La idea de atención (intuición)
Cada elemento de la secuencia "mira" a los demás y decide **a cuáles prestar atención**, usando tres vectores por elemento: *query* (qué busco), *key* (qué ofrezco), *value* (qué entrego).

$$ \text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V $$

El producto $QK^\top$ mide **similitud** entre elementos; softmax la convierte en pesos; el resultado es un promedio ponderado de los *values*. Lo veremos en detalle en el Módulo 3.

In [ ]:
# Demo conceptual: self-attention sobre 3 "tokens" (sin entrenar, solo para ver el mecanismo)
torch.manual_seed(1)
d = 4
X = torch.randn(3, d)          # 3 tokens, dimensión d
Wq, Wk, Wv = (torch.randn(d, d) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv
scores = (Q @ K.T) / d**0.5    # similitud token-a-token
weights = torch.softmax(scores, dim=-1)
out = weights @ V
print("Pesos de atención (cada fila suma 1):")
print(torch.round(weights, decimals=2))
print("\nCada token de salida es un promedio ponderado de los values segun esos pesos.")

## Cierre

Hoy vimos la **receta universal** del deep learning: datos + modelo + pérdida + optimización, y cómo el aprendizaje ocurre vía gradiente descendente y backpropagation. También el salto del perceptrón al MLP y un primer vistazo a la atención.

**En la clase práctica (Clase 2)** van a *implementar* todo esto en PyTorch. **En el Módulo 2** pasamos a visión computacional: redes convolucionales y Vision Transformers.